In [110]:
import pandas as pd

In [111]:
from sklearn.model_selection import GroupShuffleSplit

In [112]:
features = [
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "order_number",
    "cart_size",
    "trigger_product_id",
    "trigger_reordered",
    "recommended_product_id",
    "recommended_department_id",
    "support",
    "confidence",
    "lift"
    ]

group_column = "order_id"
target = "label"

In [113]:
def features_targets(data):

    X = data[features].copy()
    y = data[target].copy()

    # interpret these values for category (as product 11341 is a product_id)
    categorical_columns = [
        "order_dow",
        "order_hour_of_day",
        "trigger_product_id",
        "recommended_product_id",
        "recommended_department_id",
    ]

    for column in categorical_columns:
        X[column] = X[column].astype("category")

    numeric_columns = [
        "days_since_prior_order",
        "order_number",
        "cart_size",
        "trigger_reordered",
        "support",
        "confidence",
        "lift",
    ]

    for column in numeric_columns:
        X[column] = pd.to_numeric(
            X[column],
            errors="coerce"
        )

    X[numeric_columns] = X[numeric_columns].fillna(0)

    # target
    y = y.astype(int)

    return X, y



In [114]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)



## Data leakage problem sorted

In [115]:
test_data = pd.read_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/test_order_data.csv")
train_data = pd.read_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/train_order_data.csv")

In [116]:
dataset = features_targets(train_data)
X_train = dataset[0]
y_train = dataset[1]

dataset_02 = features_targets(test_data)
X_test = dataset_02[0]
y_test = dataset_02[1]
print(f'The dataset is loaded with {X_train.shape[0]} rows and {X_train.shape[1]} columns.')

The dataset is loaded with 213847 rows and 12 columns.


In [117]:
model_2 = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
)

model_2.fit(X_train, y_train)
print("Model training completed.")

[LightGBM] [Info] Number of positive: 19220, number of negative: 194627
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008106 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 299
[LightGBM] [Info] Number of data points in the train set: 213847, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.089877 -> initscore=-2.315134
[LightGBM] [Info] Start training from score -2.315134
Model training completed.


In [118]:
y_prediction = model_2.predict(X_test) 
y_probability = model_2.predict_proba(X_test)[:, 1] # model strong positive score b/w [0-1]

print("Predictions generated successfully.")

print("\nFirst 10 predictions:")
print(y_prediction[:10])

print("\nFirst 10 probabilities:")
print(y_probability[:10])

print("\nFirst 10 actual labels:")
print(y_test.iloc[:10].values)

Predictions generated successfully.

First 10 predictions:
[0 0 0 0 0 0 0 0 0 0]

First 10 probabilities:
[0.11199289 0.14814833 0.15893487 0.08099738 0.03304343 0.02276446
 0.02795349 0.01907999 0.01593833 0.01847887]

First 10 actual labels:
[0 0 0 0 0 0 1 0 0 0]


In [119]:
y_prediction = (y_probability >= 0.5).astype(int)


# Evaluation

accuracy = accuracy_score(y_test, y_prediction)
precision = precision_score(y_test, y_prediction)
recall = recall_score(y_test, y_prediction)
f1 = f1_score(y_test, y_prediction)
roc_auc = roc_auc_score(y_test, y_probability)


print("\nModel Evaluation")
print("----------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


Model Evaluation
----------------
Accuracy : 0.9105
Precision: 0.2000
Recall   : 0.0002
F1 Score : 0.0004
ROC-AUC  : 0.6393


In [120]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

y_probability = model_2.predict_proba(X_test)[:, 1]

y_prediction = (y_probability >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_prediction)
precision = precision_score(y_test, y_prediction, zero_division=0)
recall = recall_score(y_test, y_prediction, zero_division=0)
f1 = f1_score(y_test, y_prediction, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

print("\nModel Evaluation")
print("----------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


Model Evaluation
----------------
Accuracy : 0.9105
Precision: 0.2000
Recall   : 0.0002
F1 Score : 0.0004
ROC-AUC  : 0.6393


In [121]:
def precision_at_k(data, y_true, y_score, k=5):

    evaluation = data.loc[
        y_true.index,
        [
            "order_id",
            "trigger_product_id",
            "recommended_product_id"
        ]
    ].copy()

    evaluation["label"] = y_true.values
    evaluation["score"] = y_score

    precisions = []

    for _, group in evaluation.groupby(
        ["order_id", "trigger_product_id"]
    ):

        # Take the top K candidates,
        # or all candidates if fewer than K exist
        top_k = group.nlargest(k, "score")

        precision = top_k["label"].mean()

        precisions.append(precision)

    return sum(precisions) / len(precisions)

In [122]:
for k in [1, 3, 5, 10]:

    score = precision_at_k(
        test_data,
        y_test,
        y_probability,
        k
    )

    print(f"Precision@{k}: {score:.4f}")

Precision@1: 0.0999
Precision@3: 0.0943
Precision@5: 0.0936
Precision@10: 0.0936


In [123]:
def recall_at_k(data, y_true, y_score, k=5):

    evaluation = data.loc[
        y_true.index,
        [
            "order_id",
            "trigger_product_id",
            "recommended_product_id"
        ]
    ].copy()

    evaluation["label"] = y_true.values
    evaluation["score"] = y_score

    recalls = []

    for _, group in evaluation.groupby(
        ["order_id", "trigger_product_id"]
    ):

        total_positive = group["label"].sum()

        # No actual recommendation to recover
        if total_positive == 0:
            continue

        top_k = group.nlargest(k, "score")

        captured_positive = top_k["label"].sum()

        recall = captured_positive / total_positive

        recalls.append(recall)

    return sum(recalls) / len(recalls)

In [124]:
for k in [1, 3, 5, 10]:

    score = recall_at_k(
        test_data,
        y_test,
        y_probability,
        k
    )

    print(f"Recall@{k}: {score:.4f}")

Recall@1: 0.5372
Recall@3: 0.9669
Recall@5: 1.0000
Recall@10: 1.0000


In [128]:
test_data.columns

Index(['order_id', 'order_dow', 'order_hour_of_day', 'days_since_prior_order',
       'order_number', 'cart_size', 'trigger_product_id',
       'trigger_product_name', 'trigger_reordered', 'recommended_product_id',
       'recommended_product_name', 'recommended_department_id',
       'recommended_department_name', 'support', 'confidence', 'lift',
       'label'],
      dtype='str')

In [129]:
def calculate_mba_precision(test_data):

    results = []

    for order_id, group in test_data.groupby("order_id"):

        # Remove duplicate recommendations
        recommendations = (
            group[
                [
                    "recommended_product_id",
                    "recommended_product_name",
                    "confidence",
                    "label"
                ]
            ]
            .drop_duplicates("recommended_product_id")
            .sort_values("confidence", ascending=False)
        )

        row = {"order_id": order_id}

        for k in [1, 3, 5, 10]:

            top_k = recommendations.head(k)

            if len(top_k) == 0:
                precision = 0
            else:
                precision = top_k["label"].sum() / len(top_k)

            row[f"precision@{k}"] = precision

        results.append(row)

    return pd.DataFrame(results)

In [130]:
mba_results = calculate_mba_precision(test_data)

print("MBA / Base Recommendation Precision")
print("------------------------------------")

for k in [1, 3, 5, 10]:
    print(
        f"Precision@{k}: "
        f"{mba_results[f'precision@{k}'].mean():.4f}"
    )

MBA / Base Recommendation Precision
------------------------------------
Precision@1: 0.0863
Precision@3: 0.0763
Precision@5: 0.0811
Precision@10: 0.0811


In [131]:
def calculate_mba_recall(test_data):

    results = []

    for order_id, group in test_data.groupby("order_id"):

        # Remove duplicate recommendations
        recommendations = (
            group[
                [
                    "recommended_product_id",
                    "confidence",
                    "label"
                ]
            ]
            .drop_duplicates("recommended_product_id")
            .sort_values("confidence", ascending=False)
        )

        # Total number of relevant/purchased products
        total_relevant = recommendations["label"].sum()

        row = {"order_id": order_id}

        for k in [1, 3, 5, 10]:

            top_k = recommendations.head(k)

            # Relevant products found in top K
            relevant_found = top_k["label"].sum()

            if total_relevant == 0:
                recall = 0
            else:
                recall = relevant_found / total_relevant

            row[f"recall@{k}"] = recall

        results.append(row)

    return pd.DataFrame(results)

In [132]:
mba_recall_results = calculate_mba_recall(test_data)

print("MBA / Base Recommendation Recall")
print("---------------------------------")

for k in [1, 3, 5, 10]:
    print(
        f"Recall@{k}: "
        f"{mba_recall_results[f'recall@{k}'].mean():.4f}"
    )

MBA / Base Recommendation Recall
---------------------------------
Recall@1: 0.0794
Recall@3: 0.1485
Recall@5: 0.2098
Recall@10: 0.2098
